In [ ]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import least_squares

# ── CONSTANTS ────────────────────────────────────────────────────────────────
Omega_m  = 0.31
Omega_de = 1.0 - Omega_m
z_eq     = 3400.0

eta_opt  = 2.0 / (np.sqrt(np.pi) * (np.log(96)**2))
A        = eta_opt / 2.0
B        = eta_opt / (2.0 * np.log(1.0 + z_eq))

z_plot   = np.linspace(0, 3, 500)
z_bins   = np.array([0.0, 0.3, 0.5, 0.7, 1.1, 1.5, 1100.0])

# ── PHYSICS FUNCTIONS ────────────────────────────────────────────────────────
def w_OF(z):
    tau = np.log(1 + z)
    return -1 - (eta_opt / 2) * (1 + np.exp(-tau))

def f_Base(z):
    return (1.0 + z)**(-3.0 * A) * np.exp(-1.5 * B * (np.log(1.0 + z))**2)

def E_Base(z):
    return np.sqrt(Omega_m * (1.0 + z)**3 + Omega_de * f_Base(z))

def D_M_Base(z):
    val, _ = quad(lambda x: 1.0 / E_Base(x), 0, z)
    return val

def f_CPL(z, w0, wa):
    return (1.0 + z)**(3.0*(1.0 + w0 + wa)) * np.exp(-3.0 * wa * (z / (1.0 + z)))

def E_CPL(z, w0, wa):
    return np.sqrt(Omega_m * (1.0 + z)**3 + Omega_de * f_CPL(z, w0, wa))

def D_M_CPL(z, w0, wa):
    val, _ = quad(lambda x: 1.0 / E_CPL(x, w0, wa), 0, z)
    return val

def w_CPL(z, w0, wa):
    return w0 + wa * (z / (1.0 + z))

def find_crossing(w0, wa):
    if wa == 0:
        return None
    ratio = (-1.0 - w0) / wa
    if 0 < ratio < 1:
        return ratio / (1.0 - ratio)
    return None

# ── MOCK DATA & FITTING ──────────────────────────────────────────────────────
mock_pure = np.array([D_M_Base(z) for z in z_bins])

def make_mock(t):
    m = np.copy(mock_pure)
    m[1] = mock_pure[1] * (1 + t * 0.005)
    m[2] = mock_pure[2] * (1 - t * 0.035)
    m[3] = mock_pure[3] * (1 - t * 0.045)
    return m

def run_fit(target):
    def residuals(params):
        w0, wa = params
        cpl_distances = np.array([D_M_CPL(z, w0, wa) for z in z_bins])
        error = cpl_distances - target
        error[-1] *= 100.0
        return error
    result = least_squares(residuals, [-1.0, 0.0], method='lm')
    return result.x[0], result.x[1]

# Precompute data to make slider interactive and instant
print("Precomputing CPL fits across dataset tension grid...")
w_values = np.array([w_OF(z) for z in z_plot])

n_steps = 50
tension_levels = np.linspace(0.0, 1.0, n_steps)
w0_grid, wa_grid = [], []

for i, t in enumerate(tension_levels):
    mock = make_mock(t)
    w0, wa = run_fit(mock)
    w0_grid.append(w0)
    wa_grid.append(wa)

w0_grid = np.array(w0_grid)
wa_grid = np.array(wa_grid)

print(f"Done. Tension 1.0 (DESI Regime) → w₀={w0_grid[-1]:.4f}, wₐ={wa_grid[-1]:.4f}")

In [ ]:
import plotly.graph_objects as go

# ── BUILD ANIMATION FRAMES ────────────────────────────────────────────────────
frames = []
for i, t in enumerate(tension_levels):
    w0, wa = w0_grid[i], wa_grid[i]
    crossing = find_crossing(w0, wa)
    cpl_w = [w_CPL(z, w0, wa) for z in z_plot]

    frame_data = [
        go.Scatter(x=z_plot, y=w_values, mode='lines', line=dict(color='#4FC3F7', width=3)),
        go.Scatter(x=z_plot, y=cpl_w, mode='lines', line=dict(color='#EF5350', width=2.2, dash='dashdot')),
        go.Scatter(
            x=[crossing] if crossing else [],
            y=[-1.0] if crossing else [],
            mode='markers', marker=dict(color='#EF5350', size=12, symbol='x', line=dict(width=2.5))
        )
    ]

    crossing_str = f"z = {crossing:.2f}" if crossing else "none"
    frame_layout = go.Layout(
        title=dict(
            text=(f'The DESI Phantom Crossing — Artifact Emergence'
                  f'<br><sup style="color:#EF5350">CPL fit:  w₀ = {w0:.4f},  wₐ = {wa:.4f}'
                  f'  │  phantom crossing: {crossing_str}</sup>'),
            font=dict(size=15), x=0.5
        )
    )
    frames.append(go.Frame(data=frame_data, layout=frame_layout, name=str(i)))

# ── BASE FIGURE ───────────────────────────────────────────────────────────────
w0_0, wa_0 = w0_grid[0], wa_grid[0]
cross_0 = find_crossing(w0_0, wa_0)
cross_str_0 = f"z = {cross_0:.2f}" if cross_0 else "none"

fig_phantom = go.Figure(
    data=[
        go.Scatter(x=z_plot, y=w_values, mode='lines', name='Oros Fundus (true)', line=dict(color='#4FC3F7', width=3)),
        go.Scatter(x=z_plot, y=[w_CPL(z, w0_0, wa_0) for z in z_plot], mode='lines', name='CPL fit', line=dict(color='#EF5350', width=2.2, dash='dashdot')),
        go.Scatter(x=[], y=[], mode='markers', name='Phantom crossing', marker=dict(color='#EF5350', size=12, symbol='x', line=dict(width=2.5)))
    ],
    frames=frames
)

# ── SLIDER ────────────────────────────────────────────────────────────────────
sliders = [{
    'active': 0, 'pad': {'t': 50}, 'len': 0.9, 'x': 0.05,
    'steps': [
        {'args': [[str(i)], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate'}],
         'label': f'{int(t*100)}%', 'method': 'animate'} for i, t in enumerate(tension_levels)
    ],
    'currentvalue': {'prefix': 'Dataset tension: ', 'visible': True, 'xanchor': 'center', 'font': dict(size=13, color='white')}
}]

# ── STATIC ELEMENTS ───────────────────────────────────────────────────────────
fig_phantom.add_hline(y=-1.0, line_dash='dash', line_color='rgba(255,255,255,0.2)', line_width=1)
fig_phantom.add_annotation(x=2.2, y=-1.018, text='w<sub>eff</sub> ≈ −1.027', showarrow=False, font=dict(color='#4FC3F7', size=11), bgcolor='rgba(0,0,0,0.6)')

# ── LAYOUT ────────────────────────────────────────────────────────────────────
fig_phantom.update_layout(
    title=dict(
        text=(f'The DESI Phantom Crossing — Artifact Emergence'
              f'<br><sup style="color:#EF5350">CPL fit:  w₀ = {w0_0:.4f},  wₐ = {wa_0:.4f}'
              f'  │  phantom crossing: {cross_str_0}</sup>'),
        font=dict(size=15), x=0.5
    ),
    xaxis=dict(title='Redshift z', showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False, range=[0, 3]),
    yaxis=dict(title='w(z)', showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False, range=[-1.65, -0.75]),
    template='plotly_dark', height=640, sliders=sliders,
    legend=dict(x=0.01, y=0.02, bgcolor='rgba(0,0,0,0.5)', bordercolor='rgba(255,255,255,0.15)', borderwidth=1, font=dict(size=11)),
    margin=dict(l=60, r=40, t=90, b=110)
)

fig_phantom.show()

In [ ]:
fig_phantom.write_html(
    "desi_phantom_crossing_interactive.html",
    include_plotlyjs=True,
    full_html=True,
    config={
        'displayModeBar': True,
        'toImageButtonOptions': {
            'format': 'png',
            'filename': 'desi_phantom_crossing',
            'height': 640,
            'width': 1200,
            'scale': 2
        }
    }
)

print("Saved: desi_phantom_crossing_interactive.html")